# 👑 GeoMAS - Telemetry Analysis & Insights

Questo notebook si connette al database della telemetria (`data/simulation_metrics.duckdb`) per estrarre le metriche globali, nazionali e relazionali salvate dall'engine.
Ti permette di tracciare andamenti continui (Time-Series) per la tua Tesi, confrontando scenari e valutando la Coerenza e il livello di Inganno (Deception) degli LLM.

In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

# Configurazione Stile
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 7)

DB_PATH = 'simulation_metrics.duckdb'

# Utility per connettersi senza bloccare il file
def query_db(query, params=None):
    with duckdb.connect(DB_PATH, read_only=True) as conn:
        return conn.execute(query, params or []).df()

## 1. Elenco delle Simulazioni
Vediamo quali run sono state registrate nel DB.

In [ ]:
simulations = query_db("SELECT DISTINCT simulation_id FROM metrics_global ORDER BY simulation_id")
print("Simulazioni Disponibili:")
display(simulations)

# Imposta l'ID della simulazione che vuoi analizzare in profondità
TARGET_SIM_ID = simulations['simulation_id'].iloc[-1] if not simulations.empty else None
print(f"\nAnalizzando Simulazione ID: {TARGET_SIM_ID}")

## 2. Metriche Globali (RQ1 & RQ4)
Andamento aggregato del mondo per valutare la stabilità e la disonestà sistemica.

In [ ]:
global_df = query_db("SELECT * FROM metrics_global WHERE simulation_id = ? ORDER BY turn", [TARGET_SIM_ID])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafico 1: Deception & Coherence Media Mondiale
sns.lineplot(data=global_df, x='turn', y='global_deception_avg', ax=axes[0], label='Deception Media', color='red', linewidth=2)
sns.lineplot(data=global_df, x='turn', y='global_coherence_avg', ax=axes[0], label='Coherence Media', color='green', linewidth=2)
axes[0].set_title("Stabilità degli Agenti (Mondo)")
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_ylabel("Score (0-1)")

# Grafico 2: Militarizzazione vs Commercio
sns.lineplot(data=global_df, x='turn', y='global_trade_volume', ax=axes[1], label='Trade Volume', color='blue')
sns.lineplot(data=global_df, x='turn', y='units_created', ax=axes[1], label='Military Spending', color='orange')
axes[1].set_title("Guns vs Butter (Spesa Globale)")
axes[1].set_ylabel("Risorse")

plt.tight_layout()
plt.show()

## 3. Metriche Nazionali Grantulari (RQ3)
Confrontiamo come le diverse strategie globali adottate dalle nazioni abbiano impattato il loro livello di sincerità e la loro potenza.

In [ ]:
nation_df = query_db("SELECT * FROM metrics_nation WHERE simulation_id = ? ORDER BY turn, nation_id", [TARGET_SIM_ID])

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Deception Score per Nazione
sns.lineplot(data=nation_df, x='turn', y='deception_overall', hue='nation_id', ax=axes[0], linewidth=2)
axes[0].set_title("Deception Score per Nazione (Tasso di bugie diplomatiche/militari)")
axes[0].set_ylim(-0.1, 1.1)

# Power Projection per Nazione
sns.lineplot(data=nation_df, x='turn', y='power_projection', hue='nation_id', ax=axes[1], linewidth=2)
axes[1].set_title("Evoluzione del Potere (Power Projection)")

plt.tight_layout()
plt.show()

## 4. Soddisfazione Pubblica (RQ1)
Impatto delle guerre (o di scenari estremi) sulla popolazione.

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(data=nation_df, x='turn', y='public_satisfaction', hue='nation_id', linewidth=2)
plt.axhline(25, color='red', linestyle='--', label='Soglia Civil Unrest')
plt.title("Andamento della Soddisfazione Pubblica nel mondo")
plt.ylabel("Soddisfazione (0-100)")
plt.legend()
plt.show()

## 5. Network Analysis delle Relazioni Diplomatiche
Visualizziamo il grafo di Trust all'ultimo turno registrato per capire i blocchi di potere.

In [ ]:
# Prendiamo i dati dell'ultimo turno della simulazione target
max_turn = global_df['turn'].max() if not global_df.empty else 0
trust_df = query_db("SELECT * FROM metrics_trust WHERE simulation_id = ? AND turn = ?", [TARGET_SIM_ID, max_turn])

if not trust_df.empty:
    G = nx.DiGraph()
    
    # Aggiungi nodi (Nazioni)
    nations = pd.concat([trust_df['observer_id'], trust_df['target_id']]).unique()
    G.add_nodes_from(nations)
    
    # Aggiungi archi (escludendo il trust verso se stessi)
    for idx, row in trust_df.iterrows():
        if row['observer_id'] != row['target_id']:
            # Colore basato sullo stato di relazione
            edge_color = 'green' if row['relationship_state'] == 'ALLIANCE' else (
                         'red' if row['relationship_state'] == 'WAR' else 'gray')
            
            G.add_edge(row['observer_id'], row['target_id'], 
                       weight=row['trust_value'] / 20.0, # Scaliamo lo spessore dell'arco
                       color=edge_color)

    # Disegno del Grafo
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)  # Layout esteticamente gradevole
    
    edges = G.edges()
    colors = [G[u][v]['color'] for u, v in edges]
    weights = [G[u][v]['weight'] for u, v in edges]
    
    nx.draw(G, pos, with_labels=True, node_color='lightblue', 
            node_size=3000, font_size=12, font_weight='bold',
            edge_color=colors, width=weights, arrows=True, arrowsize=20)
    
    plt.title(f"Mappa delle Relazioni (Turno {max_turn})")
    # Legenda custom
    import matplotlib.lines as mlines
    green_line = mlines.Line2D([], [], color='green', label='Alliance/High Trust')
    gray_line = mlines.Line2D([], [], color='gray', label='Peace/Neutral')
    red_line = mlines.Line2D([], [], color='red', label='War/Hostile')
    plt.legend(handles=[green_line, gray_line, red_line], loc='lower left')
    plt.show()
else:
    print("Nessun dato di Trust trovato per l'ultimo turno.")